# RecallRelay — Promise & Supply Intelligence AI Lab

**Goal:** Recall 업무에서 고객에게 전달된 자연어 약속을 구조화하고, 현재 공급 현실과 분리하여 검증 가능한 AI 기능으로 연결한다.

### Design Principle
- LLM: 자연어 의미 해석, 고객 약속 추출
- Rule/Code: 날짜 비교, 상태 판단, Supply Risk 계산
- Evidence: AI 결과를 사람이 원문과 검증할 수 있도록 유지
- AI는 DB 상태나 Next Action을 임의로 변경하지 않는다.

### Experiment Flow
Ground Truth → Prompt v1 → Prompt v2 → Prompt v3 → JSON Parsing → 3×3 Test → Human Evaluation

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

OpenAI API Key: ··········


In [3]:
print("API Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

API Key loaded: True


In [4]:
!pip install -q openai pandas

In [6]:
from openai import OpenAI
import os
import json
import re
import pandas as pd
from datetime import datetime

client = OpenAI()
MODEL = "gpt-5.6"

## Feature 1 — Promise Integrity

### Step 0. Ground Truth

AI 결과를 보기 전에 사람이 먼저 정답 Case를 정의한다.

- **Case A — ALIGNED:** Part가 이미 도착했고 Sep 3 Appointment가 확인된 상태
- **Case B — WATCH:** Sep 3 이전 도착 안내가 있었지만 Part는 아직 IN_TRANSIT
- **Case C — AT_RISK:** 고객에게 전달된 기대는 Sep 3 이전이지만 현재 ETA는 Sep 6

**중요:** 고객 약속의 존재 여부와 현재 공급 위험도는 서로 다른 문제다.  
LLM은 자연어 약속을 추출하고, 공급 위험은 이후 Rule로 판단한다.

In [8]:
cases = {
    "A": {
        "requested_date": "2026-09-03",
        "part_status": "ARRIVED",
        "eta": "2026-09-02",
        "appointment": "2026-09-03",
        "interaction": "Part has arrived. Customer confirmed Sep 3 appointment."
    },
    "B": {
        "requested_date": "2026-09-03",
        "part_status": "IN_TRANSIT",
        "eta": "2026-09-02",
        "appointment": "2026-09-03",
        "interaction": "Customer was advised that the part should arrive before Sep 3."
    },
    "C": {
        "requested_date": "2026-09-03",
        "part_status": "IN_TRANSIT",
        "eta": "2026-09-06",
        "appointment": "2026-09-03",
        "interaction": "Customer was advised that the part should arrive before Sep 3."
    }
}

In [9]:
cases

{'A': {'requested_date': '2026-09-03',
  'part_status': 'ARRIVED',
  'eta': '2026-09-02',
  'appointment': '2026-09-03',
  'interaction': 'Part has arrived. Customer confirmed Sep 3 appointment.'},
 'B': {'requested_date': '2026-09-03',
  'part_status': 'IN_TRANSIT',
  'eta': '2026-09-02',
  'appointment': '2026-09-03',
  'interaction': 'Customer was advised that the part should arrive before Sep 3.'},
 'C': {'requested_date': '2026-09-03',
  'part_status': 'IN_TRANSIT',
  'eta': '2026-09-06',
  'appointment': '2026-09-03',
  'interaction': 'Customer was advised that the part should arrive before Sep 3.'}}

### Step 1. Prompt v1 — Zero-shot Baseline

먼저 업무 기준을 거의 주지 않은 단순 Prompt를 사용한다.

**확인할 것**
- 고객 약속을 제대로 찾는가?
- 없는 날짜나 사실을 만들어내는가?
- 근거 문장을 보여주는가?
- 코드에서 재사용 가능한 구조로 출력하는가?

이 단계의 목적은 처음부터 좋은 Prompt를 만드는 것이 아니라, 기준이 부족할 때 어떤 문제가 생기는지 확인하는 것이다.

In [10]:
def build_prompt_v1(case):
    return f"""
Extract any customer commitment from the interaction below.

INTERACTION:
{case["interaction"]}
"""

In [11]:
prompt_v1 = build_prompt_v1(cases["B"])
print(prompt_v1)


Extract any customer commitment from the interaction below.

INTERACTION:
Customer was advised that the part should arrive before Sep 3.



In [12]:
def call_model(prompt):
    response = client.responses.create(
        model=MODEL,
        input=prompt
    )
    return response.output_text

In [14]:
raw_v1_B = call_model(prompt_v1)
print(raw_v1_B)

The part is expected to arrive before September 3.


### Step 2. Prompt v2 — Positive / Negative Few-shot

v1보다 업무 기준을 명확하게 하기 위해 정답 예시를 제공한다.

- Positive: 실제 고객 약속
- Negative: 단순 시스템 ETA

**가설:** Few-shot을 제공하면 ETA Fact와 Customer Commitment를 더 잘 구분할 수 있을 것이다.

In [15]:
def build_prompt_v2(case):
    return f"""
You extract CUSTOMER COMMITMENTS only.

Positive example:
Input: "We will call you Friday afternoon."
Output: {{"exists": true, "type": "CALLBACK", "date_or_condition": "Friday afternoon"}}

Negative example:
Input: "Current ETA is Sep 2."
Output: {{"exists": false, "type": null, "date_or_condition": null}}

Now analyze:
{case["interaction"]}
"""

In [16]:
prompt_v2 = build_prompt_v2(cases["B"])
print(prompt_v2)


You extract CUSTOMER COMMITMENTS only.

Positive example:
Input: "We will call you Friday afternoon."
Output: {"exists": true, "type": "CALLBACK", "date_or_condition": "Friday afternoon"}

Negative example:
Input: "Current ETA is Sep 2."
Output: {"exists": false, "type": null, "date_or_condition": null}

Now analyze:
Customer was advised that the part should arrive before Sep 3.



In [17]:
raw_v2_B = call_model(prompt_v2)
print(raw_v2_B)

{"exists": false, "type": null, "date_or_condition": null}


#### Observation — v2 False Negative

Case B에서 모델은 `exists: false`를 반환했다.

하지만 `"Customer was advised that the part should arrive before Sep 3."`는 단순 시스템 ETA가 아니라 실제 고객에게 전달된 내용이다.

**Failure Hypothesis:**  
Negative Few-shot의 `"Current ETA is Sep 2."` 예시가 너무 강하게 작용해, ETA와 관련된 고객 안내까지 단순 Fact로 일반화한 것으로 보인다.

→ 다음 버전에서는 Current Facts, 업무 범위, Evidence 요구 조건을 추가한다.

### Step 3. Prompt v3 — Context + Guardrails + Evidence

v2의 False Negative를 보완하기 위해 다음 정보를 추가한다.

- Current Case Facts
- 명확한 Role과 Task
- Hallucination 방지 Constraint
- Evidence 원문 반환
- JSON 출력 형식
- Supply Alignment / Next Action 변경 금지

**목표:** 자연어 약속 추출은 LLM에 맡기되, 운영 상태 판단까지 모델이 확장하지 못하도록 범위를 제한한다.

In [18]:
def build_prompt_v3(case):
    return f"""
ROLE: Service Manager

TASK:
Extract only customer-facing commitments from the interaction.

CURRENT FACTS
- Requested date: {case["requested_date"]}
- Part status: {case["part_status"]}
- ETA: {case["eta"]}
- Appointment: {case["appointment"]}

INTERACTION
{case["interaction"]}

CONSTRAINTS
1. Do not invent a commitment.
2. If evidence is insufficient, return null fields.
3. Evidence must be copied from the interaction.
4. Do not change Supply Alignment.
5. Do not create a Next Action.

OUTPUT JSON ONLY
{{
  "exists": true/false,
  "commitment_type": "string or null",
  "date_or_condition": "string or null",
  "evidence": "string or null"
}}
"""

In [19]:
prompt_v3 = build_prompt_v3(cases["B"])
print(prompt_v3)


ROLE: Service Manager

TASK:
Extract only customer-facing commitments from the interaction.

CURRENT FACTS
- Requested date: 2026-09-03
- Part status: IN_TRANSIT
- ETA: 2026-09-02
- Appointment: 2026-09-03

INTERACTION
Customer was advised that the part should arrive before Sep 3.

CONSTRAINTS
1. Do not invent a commitment.
2. If evidence is insufficient, return null fields.
3. Evidence must be copied from the interaction.
4. Do not change Supply Alignment.
5. Do not create a Next Action.

OUTPUT JSON ONLY
{
  "exists": true/false,
  "commitment_type": "string or null",
  "date_or_condition": "string or null",
  "evidence": "string or null"
}



In [20]:
raw_v3_B = call_model(prompt_v3)
print(raw_v3_B)

{
  "exists": true,
  "commitment_type": "Part arrival",
  "date_or_condition": "Before Sep 3",
  "evidence": "Customer was advised that the part should arrive before Sep 3."
}


#### Observation — v3 Improvement

Case B에서 `exists: true`로 수정되었고 다음 정보가 구조화되었다.

- Commitment Type: Part arrival
- Date / Condition: Before Sep 3
- Evidence: Interaction 원문

v2의 False Negative가 Context와 Guardrail을 추가한 뒤 개선되었다.

### Step 4. JSON Validation

사람이 읽었을 때 맞아 보이는 답변만으로는 서비스에 연결할 수 없다.

AI 응답이 실제 Python 코드에서 JSON으로 파싱 가능한지 확인한다.

**Success Condition:** `Parse Error = None`

In [21]:
def extract_json(text):
    cleaned = text.strip()
    cleaned = re.sub(
        r"^```json\s*|^```\s*|```$",
        "",
        cleaned,
        flags=re.MULTILINE
    ).strip()

    try:
        return json.loads(cleaned), None
    except Exception as e:
        return None, str(e)

In [22]:
parsed_v3_B, parse_error_v3_B = extract_json(raw_v3_B)

print("Parsed:")
print(parsed_v3_B)

print("\nParse Error:")
print(parse_error_v3_B)

Parsed:
{'exists': True, 'commitment_type': 'Part arrival', 'date_or_condition': 'Before Sep 3', 'evidence': 'Customer was advised that the part should arrive before Sep 3.'}

Parse Error:
None


### Step 5. 3 Cases × 3 Prompt Versions

동일한 Ground Truth에 대해 v1, v2, v3를 각각 실행한다.

총 **9회 API 결과**를 저장해 Prompt 변경에 따른 차이를 비교한다.

비교 항목:
- Commitment 판단
- 날짜 / 조건 판단
- Evidence
- Hallucination
- Unauthorized Action
- JSON Validity

In [23]:
def build_prompt(version, case):
    if version == "v1":
        return build_prompt_v1(case)
    elif version == "v2":
        return build_prompt_v2(case)
    elif version == "v3":
        return build_prompt_v3(case)
    else:
        raise ValueError(f"Unknown prompt version: {version}")

In [24]:
results = []

for case_id, case in cases.items():
    for version in ["v1", "v2", "v3"]:
        prompt = build_prompt(version, case)
        raw = call_model(prompt)
        parsed, err = extract_json(raw)

        results.append({
            "case_id": case_id,
            "prompt_version": version,
            "raw_response": raw,
            "json_valid": err is None,
            "parse_error": err
        })

df = pd.DataFrame(results)
df

,case_id,prompt_version,raw_response,json_valid,parse_error
0,A,v1,Customer confirmed an appointment for Septembe...,False,Expecting value: line 1 column 1 (char 0)
1,A,v2,"{""exists"": true, ""type"": ""APPOINTMENT"", ""date_...",True,None
2,A,v3,"{\n ""exists"": true,\n ""commitment_type"": ""Ap...",True,None
3,B,v1,The part should arrive before September 3.,False,Expecting value: line 1 column 1 (char 0)
4,B,v2,"{""exists"": false, ""type"": null, ""date_or_condi...",True,None
5,B,v3,"{\n ""exists"": true,\n ""commitment_type"": ""Pa...",True,None
6,C,v1,No customer commitment was made.,False,Expecting value: line 1 column 1 (char 0)
7,C,v2,"{""exists"": false, ""type"": null, ""date_or_condi...",True,None
8,C,v3,"{\n ""exists"": true,\n ""commitment_type"": ""Pa...",True,None


In [25]:
df.to_csv("prompt_experiment.csv", index=False)
print("Saved:", len(df), "rows")

Saved: 9 rows


In [26]:
pd.set_option("display.max_colwidth", None)

for _, row in df.iterrows():
    print("=" * 70)
    print("CASE:", row["case_id"], "| PROMPT:", row["prompt_version"])
    print(row["raw_response"])
    print()

CASE: A | PROMPT: v1
Customer confirmed an appointment for September 3.

CASE: A | PROMPT: v2
{"exists": true, "type": "APPOINTMENT", "date_or_condition": "Sep 3"}

CASE: A | PROMPT: v3
{
  "exists": true,
  "commitment_type": "Appointment",
  "date_or_condition": "Sep 3",
  "evidence": "Customer confirmed Sep 3 appointment."
}

CASE: B | PROMPT: v1
The part should arrive before September 3.

CASE: B | PROMPT: v2
{"exists": false, "type": null, "date_or_condition": null}

CASE: B | PROMPT: v3
{
  "exists": true,
  "commitment_type": "Part arrival",
  "date_or_condition": "Before Sep 3",
  "evidence": "the part should arrive before Sep 3"
}

CASE: C | PROMPT: v1
No customer commitment was made.

CASE: C | PROMPT: v2
{"exists": false, "type": null, "date_or_condition": null}

CASE: C | PROMPT: v3
{
  "exists": true,
  "commitment_type": "Part arrival",
  "date_or_condition": "before Sep 3",
  "evidence": "the part should arrive before Sep 3"
}



### Step 6. Human Evaluation

LLM이 자기 결과를 평가하게 하지 않고, 미리 정의한 Ground Truth를 기준으로 사람이 직접 평가한다.

특히 다음을 분리해서 본다.

- 의미가 맞는가?
- 날짜 / 조건이 맞는가?
- 원문 Evidence가 있는가?
- 없는 사실을 생성했는가?
- 업무 권한 밖의 Action을 만들었는가?
- 코드에서 재사용 가능한 JSON인가?

In [27]:
human_eval = pd.DataFrame([
    ["A", "v1", True,  True,  False, False, False, False,
     "Semantically correct, but free-text output and no explicit evidence."],

    ["A", "v2", True,  True,  False, False, False, True,
     "Correct structured extraction, but prompt does not require evidence."],

    ["A", "v3", True,  True,  True,  False, False, True,
     "Correct commitment, date, evidence, and structured JSON."],

    ["B", "v1", True,  True,  False, False, False, False,
     "Commitment meaning captured, but output is unstructured."],

    ["B", "v2", False, False, False, False, False, True,
     "False negative: customer-facing part arrival commitment missed."],

    ["B", "v3", True,  True,  True,  False, False, True,
     "False negative corrected after adding context, constraints, and evidence."],

    ["C", "v1", False, False, False, False, False, False,
     "False negative. Same interaction as Case B produced inconsistent judgment."],

    ["C", "v2", False, False, False, False, False, True,
     "JSON format improved, but customer-facing commitment still missed."],

    ["C", "v3", True,  True,  True,  False, False, True,
     "Correctly preserves commitment despite changed supply facts."]
], columns=[
    "case_id",
    "prompt_version",
    "commitment_correct",
    "date_correct",
    "evidence_supported",
    "hallucination",
    "unauthorized_action",
    "json_valid",
    "notes"
])

human_eval

,case_id,prompt_version,commitment_correct,date_correct,evidence_supported,hallucination,unauthorized_action,json_valid,notes
0,A,v1,True,True,False,False,False,False,"Semantically correct, but free-text output and no explicit evidence."
1,A,v2,True,True,False,False,False,True,"Correct structured extraction, but prompt does not require evidence."
2,A,v3,True,True,True,False,False,True,"Correct commitment, date, evidence, and structured JSON."
3,B,v1,True,True,False,False,False,False,"Commitment meaning captured, but output is unstructured."
4,B,v2,False,False,False,False,False,True,False negative: customer-facing part arrival commitment missed.
5,B,v3,True,True,True,False,False,True,"False negative corrected after adding context, constraints, and evidence."
6,C,v1,False,False,False,False,False,False,False negative. Same interaction as Case B produced inconsistent judgment.
7,C,v2,False,False,False,False,False,True,"JSON format improved, but customer-facing commitment still missed."
8,C,v3,True,True,True,False,False,True,Correctly preserves commitment despite changed supply facts.


In [28]:
human_eval.to_csv("prompt_human_evaluation.csv", index=False)

print("Saved:", len(human_eval), "evaluations")

Saved: 9 evaluations


In [29]:
summary = human_eval.groupby("prompt_version").agg(
    commitment_accuracy=("commitment_correct", "mean"),
    date_accuracy=("date_correct", "mean"),
    evidence_rate=("evidence_supported", "mean"),
    json_valid_rate=("json_valid", "mean")
)

summary = (summary * 100).round(1)
summary

,commitment_accuracy,date_accuracy,evidence_rate,json_valid_rate
prompt_version,,,,
v1,66.7,66.7,0.0,0.0
v2,33.3,33.3,0.0,100.0
v3,100.0,100.0,100.0,100.0


### Experiment Result

초기 3개 Ground Truth Case 기준:

| Prompt | Commitment | Date | Evidence | JSON |
|---|---:|---:|---:|---:|
| v1 | 66.7% | 66.7% | 0% | 0% |
| v2 | 33.3% | 33.3% | 0% | 100% |
| v3 | 100% | 100% | 100% | 100% |

### Interpretation

v2는 JSON 형식은 안정화했지만 Negative Few-shot을 과도하게 일반화해 False Negative가 발생했다.

v3에서는 Current Facts, Constraints, Evidence를 추가하면서 초기 3개 Case 모두 사전에 정의한 평가 기준을 충족했다.

**주의:** 3개 통제 Case의 초기 실험 결과이며 Production Accuracy를 의미하지 않는다.

In [30]:
summary.to_csv("prompt_version_summary.csv")
print("Saved prompt_version_summary.csv")

Saved prompt_version_summary.csv


In [31]:
with open("promise_v1_zero.txt", "w") as f:
    f.write(build_prompt_v1(cases["B"]))

with open("promise_v2_fewshot.txt", "w") as f:
    f.write(build_prompt_v2(cases["B"]))

with open("promise_v3_context_guardrails.txt", "w") as f:
    f.write(build_prompt_v3(cases["B"]))

print("Saved 3 prompt files")

Saved 3 prompt files


In [32]:
with open("ground_truth_cases.json", "w") as f:
    json.dump(cases, f, indent=2)

print("Saved ground_truth_cases.json")

Saved ground_truth_cases.json


In [33]:
import zipfile

files_to_save = [
    "prompt_experiment.csv",
    "prompt_human_evaluation.csv",
    "prompt_version_summary.csv",
    "promise_v1_zero.txt",
    "promise_v2_fewshot.txt",
    "promise_v3_context_guardrails.txt",
    "ground_truth_cases.json"
]

with zipfile.ZipFile("recallrelay_day1_prompt_lab.zip", "w") as zipf:
    for file in files_to_save:
        zipf.write(file)

print("Created recallrelay_day1_prompt_lab.zip")

Created recallrelay_day1_prompt_lab.zip


In [34]:
from google.colab import files
files.download("recallrelay_day1_prompt_lab.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

# Feature 2 — Overpromise Guard

## Problem

현재 공급 데이터가 아직 확정되지 않았는데 직원이 고객에게 `"will arrive"`, `"guaranteed"`처럼 확정적인 표현을 사용할 수 있다.

이 기능의 목적은 문장을 대신 작성하는 것이 아니라, **현재 공급 Facts보다 과도하게 확정적인 고객 안내 표현을 검증하는 것**이다.

### Input → Process → Output

Customer Draft  
→ Current Supply Facts  
→ LLM Communication Risk Review  
→ Evidence  
→ Safer Wording  
→ Employee Decision

### Design Boundary

AI는 Supply Alignment나 Next Action을 변경하지 않는다.  
최종 고객 안내 여부는 직원이 판단한다.

## Step 1. Communication Risk Prompt

고객에게 보낼 문장과 현재 공급 Facts를 함께 제공하고,
문장의 확정성이 실제 공급 근거를 넘어서는지 검사한다.

### AI가 판단할 것
- 공급 사실보다 과도하게 확정적인 표현인지
- 어떤 표현이 문제인지
- 왜 현재 Facts로는 해당 표현을 보장할 수 없는지
- 같은 Facts 범위 안에서 더 안전한 표현은 무엇인지

### AI가 하지 않을 것
- Supply Alignment 변경
- Next Action 결정
- DB 값 변경
- 현재 Facts에 없는 정보 추가

In [35]:
def build_overpromise_prompt(
    draft,
    part_status,
    eta,
    arrived,
    appointment,
    supply_alignment
):
    return f"""
ROLE: Communication risk reviewer for a recall service team.

CURRENT FACTS
- Part status: {part_status}
- ETA: {eta}
- Physical arrival confirmed: {arrived}
- Appointment: {appointment}
- Supply alignment: {supply_alignment}

DRAFT TO CUSTOMER
{draft}

TASK
Find statements that are more certain than the facts support.
Do not change operational status or next action.
Do not add information that is not supported by CURRENT FACTS.

OUTPUT JSON ONLY
{{
  "risk": "NONE|LOW|HIGH",
  "unsupported_phrase": "string or null",
  "reason": "string or null",
  "safer_wording": "string or null",
  "evidence_facts": ["string"]
}}
"""

## Step 2. Ground Truth Test Cases

LLM 결과를 보기 전에 사람이 먼저 문장의 기대 위험 수준을 정의한다.

| Case | Customer Draft | Supply Reality | Expected |
|---|---|---|---|
| OP1 | "will arrive" | IN_TRANSIT, ETA only | Warning |
| OP2 | "expected to arrive" | IN_TRANSIT, ETA exists | Safe or Low |
| OP3 | "has arrived" | Physical arrival confirmed | Safe |
| OP4 | "appointment is guaranteed" | Part not physically received | Warning |

### Test Purpose

단순히 위험한 문장을 찾는 것만 확인하지 않는다.

안전한 표현까지 모두 위험하다고 판단하는 **과잉 경고 False Positive**도 함께 확인한다.

In [36]:
overpromise_cases = [
    {
        "id": "OP1",
        "draft": "Your part will arrive on Sep 2.",
        "part_status": "IN_TRANSIT",
        "eta": "2026-09-02",
        "arrived": "NO",
        "appointment": "2026-09-03",
        "supply_alignment": "WATCH",
        "expected": "WARNING"
    },
    {
        "id": "OP2",
        "draft": "Your part is expected to arrive on Sep 2.",
        "part_status": "IN_TRANSIT",
        "eta": "2026-09-02",
        "arrived": "NO",
        "appointment": "2026-09-03",
        "supply_alignment": "WATCH",
        "expected": "SAFE_OR_LOW"
    },
    {
        "id": "OP3",
        "draft": "Your part has arrived.",
        "part_status": "ARRIVED",
        "eta": "2026-09-02",
        "arrived": "YES",
        "appointment": "2026-09-03",
        "supply_alignment": "ALIGNED",
        "expected": "SAFE"
    },
    {
        "id": "OP4",
        "draft": "Your Sep 3 appointment is guaranteed.",
        "part_status": "IN_TRANSIT",
        "eta": "2026-09-02",
        "arrived": "NO",
        "appointment": "2026-09-03",
        "supply_alignment": "WATCH",
        "expected": "WARNING"
    }
]

## Step 3. Run the Same Review Across 4 Cases

같은 Prompt와 판단 기준을 유지한 상태에서
공급 Facts와 고객 Draft만 변경해 4개 Case를 비교한다.

### Evaluation Questions
1. 확정적인 "will arrive"를 위험으로 탐지하는가?
2. "expected to arrive"까지 과도하게 경고하지 않는가?
3. 실제 ARRIVED 상태에서는 "has arrived"를 허용하는가?
4. 물리 입고 전 "guaranteed" 표현을 위험으로 판단하는가?
5. safer_wording이 새로운 사실을 만들어내지 않는가?
6. evidence_facts가 실제 입력 Facts에 근거하는가?

In [37]:
overpromise_results = []

for case in overpromise_cases:
    prompt = build_overpromise_prompt(
        draft=case["draft"],
        part_status=case["part_status"],
        eta=case["eta"],
        arrived=case["arrived"],
        appointment=case["appointment"],
        supply_alignment=case["supply_alignment"]
    )

    raw = call_model(prompt)
    parsed, err = extract_json(raw)

    overpromise_results.append({
        "case_id": case["id"],
        "draft": case["draft"],
        "expected": case["expected"],
        "raw_response": raw,
        "json_valid": err is None,
        "parse_error": err
    })

overpromise_df = pd.DataFrame(overpromise_results)
overpromise_df

,case_id,draft,expected,raw_response,json_valid,parse_error
0,OP1,Your part will arrive on Sep 2.,WARNING,"{\n ""risk"": ""HIGH"",\n ""unsupported_phrase"": ""will arrive on Sep 2"",\n ""reason"": ""Sep 2 is an estimated arrival date, and physical arrival has not been confirmed."",\n ""safer_wording"": ""Your part is expected to arrive on Sep 2."",\n ""evidence_facts"": [\n ""Part status: IN_TRANSIT"",\n ""ETA: 2026-09-02"",\n ""Physical arrival confirmed: NO"",\n ""Supply alignment: WATCH""\n ]\n}",True,None
1,OP2,Your part is expected to arrive on Sep 2.,SAFE_OR_LOW,"{\n ""risk"": ""NONE"",\n ""unsupported_phrase"": null,\n ""reason"": null,\n ""safer_wording"": null,\n ""evidence_facts"": [\n ""Part status: IN_TRANSIT"",\n ""ETA: 2026-09-02""\n ]\n}",True,None
2,OP3,Your part has arrived.,SAFE,"{""risk"":""NONE"",""unsupported_phrase"":null,""reason"":null,""safer_wording"":null,""evidence_facts"":[""Part status: ARRIVED"",""Physical arrival confirmed: YES""]}",True,None
3,OP4,Your Sep 3 appointment is guaranteed.,WARNING,"{\n ""risk"": ""HIGH"",\n ""unsupported_phrase"": ""is guaranteed"",\n ""reason"": ""The appointment cannot be guaranteed because the part is still in transit, physical arrival is not confirmed, and supply alignment is on watch."",\n ""safer_wording"": ""Your Sep 3 appointment is scheduled."",\n ""evidence_facts"": [\n ""Part status: IN_TRANSIT"",\n ""ETA: 2026-09-02"",\n ""Physical arrival confirmed: NO"",\n ""Appointment: 2026-09-03"",\n ""Supply alignment: WATCH""\n ]\n}",True,None


## Step 4. Inspect Raw Responses

표의 성공 여부만 보지 않고 Raw Response를 직접 확인한다.

특히 다음 실패 유형을 찾는다.

- **False Negative:** 위험한 표현을 안전하다고 판단
- **False Positive:** 안전한 표현까지 위험하다고 판단
- **Unsupported Rewrite:** safer wording에 없는 사실을 추가
- **Weak Evidence:** 판단 근거가 Current Facts와 연결되지 않음
- **Invalid JSON:** 서비스 코드가 결과를 안정적으로 사용할 수 없음

In [38]:
for _, row in overpromise_df.iterrows():
    print("=" * 70)
    print("CASE:", row["case_id"])
    print("DRAFT:", row["draft"])
    print("EXPECTED:", row["expected"])
    print("RESULT:")
    print(row["raw_response"])
    print()

CASE: OP1
DRAFT: Your part will arrive on Sep 2.
EXPECTED: WARNING
RESULT:
{
  "risk": "HIGH",
  "unsupported_phrase": "will arrive on Sep 2",
  "reason": "Sep 2 is an estimated arrival date, and physical arrival has not been confirmed.",
  "safer_wording": "Your part is expected to arrive on Sep 2.",
  "evidence_facts": [
    "Part status: IN_TRANSIT",
    "ETA: 2026-09-02",
    "Physical arrival confirmed: NO",
    "Supply alignment: WATCH"
  ]
}

CASE: OP2
DRAFT: Your part is expected to arrive on Sep 2.
EXPECTED: SAFE_OR_LOW
RESULT:
{
  "risk": "NONE",
  "unsupported_phrase": null,
  "reason": null,
  "safer_wording": null,
  "evidence_facts": [
    "Part status: IN_TRANSIT",
    "ETA: 2026-09-02"
  ]
}

CASE: OP3
DRAFT: Your part has arrived.
EXPECTED: SAFE
RESULT:
{"risk":"NONE","unsupported_phrase":null,"reason":null,"safer_wording":null,"evidence_facts":["Part status: ARRIVED","Physical arrival confirmed: YES"]}

CASE: OP4
DRAFT: Your Sep 3 appointment is guaranteed.
EXPECTED: 

## Step 5. Observation & Evaluation

### Result

4개의 통제 Test Case에서 기대한 Communication Risk 구분이 모두 확인되었다.

- **OP1 — "will arrive"**
  - Result: `HIGH`
  - 실제 상태는 `IN_TRANSIT`이며 Physical Arrival이 확인되지 않았으므로 확정 표현을 경고했다.
  - `"expected to arrive"`라는 더 낮은 확정성의 표현을 제안했다.

- **OP2 — "expected to arrive"**
  - Result: `NONE`
  - ETA가 존재하는 상태에서 예상 표현을 사용했으므로 과잉 경고하지 않았다.

- **OP3 — "has arrived"**
  - Result: `NONE`
  - Part Status=`ARRIVED`, Physical Arrival=`YES`라는 실제 근거가 있으므로 허용했다.

- **OP4 — "appointment is guaranteed"**
  - Result: `HIGH`
  - Part가 아직 `IN_TRANSIT`이고 Physical Arrival이 확인되지 않았으므로 `guaranteed`를 과도한 확정 표현으로 판단했다.

### What Worked

- 위험한 확정 표현과 안전한 예상 표현을 구분했다.
- 안전한 문장까지 모두 경고하는 False Positive가 발생하지 않았다.
- `safer_wording`은 입력된 Current Facts 범위를 벗어난 새로운 사실을 추가하지 않았다.
- 판단 이유와 `evidence_facts`를 함께 반환해 사람이 결과를 검증할 수 있었다.
- Supply Alignment나 Next Action을 변경하지 않았다.

### Limitation

이 결과는 4개의 통제된 Test Case에 대한 초기 Prototype 검증이다.
실제 운영 환경의 다양한 고객 표현에 대한 일반적인 성능을 의미하지 않는다.

In [39]:
overpromise_eval_rows = []

for case, result in zip(overpromise_cases, overpromise_results):
    parsed, err = extract_json(result["raw_response"])

    actual_risk = parsed.get("risk") if parsed else None

    if case["id"] == "OP1":
        risk_correct = actual_risk == "HIGH"
    elif case["id"] == "OP2":
        risk_correct = actual_risk in ["NONE", "LOW"]
    elif case["id"] == "OP3":
        risk_correct = actual_risk == "NONE"
    elif case["id"] == "OP4":
        risk_correct = actual_risk == "HIGH"
    else:
        risk_correct = False

    overpromise_eval_rows.append({
        "case_id": case["id"],
        "draft": case["draft"],
        "expected": case["expected"],
        "actual_risk": actual_risk,
        "risk_correct": risk_correct,
        "json_valid": err is None,
        "unsupported_phrase": parsed.get("unsupported_phrase") if parsed else None,
        "safer_wording": parsed.get("safer_wording") if parsed else None,
        "evidence_facts": parsed.get("evidence_facts") if parsed else None
    })

overpromise_eval = pd.DataFrame(overpromise_eval_rows)
overpromise_eval

,case_id,draft,expected,actual_risk,risk_correct,json_valid,unsupported_phrase,safer_wording,evidence_facts
0,OP1,Your part will arrive on Sep 2.,WARNING,HIGH,True,True,will arrive on Sep 2,Your part is expected to arrive on Sep 2.,"[Part status: IN_TRANSIT, ETA: 2026-09-02, Physical arrival confirmed: NO, Supply alignment: WATCH]"
1,OP2,Your part is expected to arrive on Sep 2.,SAFE_OR_LOW,NONE,True,True,None,None,"[Part status: IN_TRANSIT, ETA: 2026-09-02]"
2,OP3,Your part has arrived.,SAFE,NONE,True,True,None,None,"[Part status: ARRIVED, Physical arrival confirmed: YES]"
3,OP4,Your Sep 3 appointment is guaranteed.,WARNING,HIGH,True,True,is guaranteed,Your Sep 3 appointment is scheduled.,"[Part status: IN_TRANSIT, ETA: 2026-09-02, Physical arrival confirmed: NO, Appointment: 2026-09-03, Supply alignment: WATCH]"


## Step 6. Initial Test Result

4개의 통제 Test Case에서 사전에 정의한 Communication Risk 기준과 일치하는 결과를 확인했다.

- Risk behavior matched: **4 / 4**
- JSON parsing succeeded: **4 / 4**
- Unsafe certainty (`will arrive`, `guaranteed`) was flagged.
- Supported wording (`expected to arrive`, confirmed `has arrived`) was not over-flagged.
- No Supply Alignment or Next Action was changed by the LLM.

### Key Finding

Overpromise Guard는 고객 문장을 대신 작성하는 기능이 아니라,
현재 Supply Facts와 고객 안내문의 확정성 사이의 차이를 검증하는 Guard 역할로 제한했다.

### Limitation

이 결과는 4개의 통제된 초기 Test Case에 한정된다.
실제 고객 커뮤니케이션 전반에 대한 성능이나 Production Accuracy를 의미하지 않는다.

In [40]:
overpromise_summary = {
    "total_cases": len(overpromise_eval),
    "expected_behavior_met": int(overpromise_eval["risk_correct"].sum()),
    "json_valid": int(overpromise_eval["json_valid"].sum())
}

print(overpromise_summary)

{'total_cases': 4, 'expected_behavior_met': 4, 'json_valid': 4}


In [41]:
overpromise_df.to_csv("overpromise_experiment.csv", index=False)
overpromise_eval.to_csv("overpromise_human_evaluation.csv", index=False)

with open("overpromise_summary.json", "w") as f:
    json.dump(overpromise_summary, f, indent=2)

with open("overpromise_guard_prompt.txt", "w") as f:
    f.write(
        build_overpromise_prompt(
            draft="{CUSTOMER_DRAFT}",
            part_status="{PART_STATUS}",
            eta="{ETA}",
            arrived="{ARRIVAL_CONFIRMED}",
            appointment="{APPOINTMENT}",
            supply_alignment="{SUPPLY_ALIGNMENT}"
        )
    )

print("Saved Overpromise Guard files")

Saved Overpromise Guard files


In [42]:
overpromise_files = [
    "overpromise_experiment.csv",
    "overpromise_human_evaluation.csv",
    "overpromise_summary.json",
    "overpromise_guard_prompt.txt"
]

with zipfile.ZipFile("recallrelay_overpromise_guard.zip", "w") as zipf:
    for file in overpromise_files:
        zipf.write(file)

files.download("recallrelay_overpromise_guard.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

# Feature 3 — Change Impact Radar

## Problem

Recall 업무에서는 ETA나 Part Status가 변경되었을 때
단순히 "값이 바뀌었다"는 사실보다 기존 고객 약속과 Appointment에
어떤 영향이 생기는지를 빠르게 확인해야 한다.

### Design Principle

**Code / Rule**
- Before / After 차이 계산
- ETA와 고객 요구일 비교
- Supply Risk 상태 계산

**LLM**
- 변경으로 영향을 받는 기존 고객 기대와 Workflow를 설명
- Evidence 기반 Summary 제공

### Boundary

LLM은 DB 값을 변경하지 않는다.
Supply Risk 변경은 자연어 판단이 아니라 Rule로 계산한다.

## Step 1. Detect the Change with Code

먼저 AI를 호출하지 않고 일반 코드로
실제 데이터에서 어떤 필드가 변경되었는지 확인한다.

**Scenario**

- Before ETA: Sep 2
- After ETA: Sep 6
- Customer Need: Sep 3
- Appointment: Sep 3
- Previous Commitment: "part should arrive before Sep 3"

예상되는 핵심 변화는 ETA 지연이다.

In [43]:
before = {
    "eta": "2026-09-02",
    "part_status": "IN_TRANSIT"
}

after = {
    "eta": "2026-09-06",
    "part_status": "IN_TRANSIT"
}

changes = {
    key: {
        "before": before.get(key),
        "after": after.get(key)
    }
    for key in after
    if before.get(key) != after.get(key)
}

print(changes)

{'eta': {'before': '2026-09-02', 'after': '2026-09-06'}}


## Step 2. Calculate Risk with Deterministic Rule

ETA가 고객에게 전달된 약속보다 늦는지는
자연어 해석이 필요한 문제가 아니라 날짜 비교 문제다.

따라서 이 판단은 LLM이 아니라 Rule로 계산한다.

- ETA <= Promised Date → WATCH
- ETA > Promised Date → AT_RISK
- Part already ARRIVED → SAFE

In [44]:
from datetime import date

def promise_risk(part_status, eta, promised_date):
    if not promised_date:
        return "NO_COMMITMENT"

    if part_status == "ARRIVED":
        return "SAFE"

    if eta is None:
        return "AT_RISK"

    eta_date = date.fromisoformat(eta)
    promise_date = date.fromisoformat(promised_date)

    return "AT_RISK" if eta_date > promise_date else "WATCH"

In [45]:
risk_before = promise_risk(
    part_status="IN_TRANSIT",
    eta="2026-09-02",
    promised_date="2026-09-03"
)

risk_after = promise_risk(
    part_status="IN_TRANSIT",
    eta="2026-09-06",
    promised_date="2026-09-03"
)

print("Before:", risk_before)
print("After :", risk_after)

Before: WATCH
After : AT_RISK


### Observation — Deterministic Risk Change

ETA가 `2026-09-02 → 2026-09-06`으로 변경되자
동일한 고객 약속일 `2026-09-03`을 기준으로 Rule 결과가 다음과 같이 변경되었다.

- Before: `WATCH`
- After: `AT_RISK`

이 Risk 변화는 LLM의 자연어 판단이 아니라
Python 날짜 비교 Rule로 계산했다.

**Design Decision:**  
정확하고 재현 가능한 날짜 및 상태 비교는 LLM에 맡기지 않는다.

## Step 3. Explain Business Impact with LLM

Code가 이미 다음 사실을 확정했다.

- ETA가 Sep 2 → Sep 6으로 변경됨
- 고객 요구일은 Sep 3
- Appointment는 Sep 3
- 기존 고객 안내는 "part should arrive before Sep 3"
- Rule Risk는 WATCH → AT_RISK로 변경됨

이제 LLM에는 Risk를 다시 판단시키지 않고,
**기존 고객 기대와 Workflow 중 무엇이 영향을 받는지 설명하는 역할만 맡긴다.**

### LLM Boundary

- DB 값을 변경하지 않는다.
- Risk 값을 다시 계산하지 않는다.
- Next Action을 생성하지 않는다.
- 입력 Facts에 없는 사실을 만들지 않는다.
- 판단 근거를 Evidence로 반환한다.

In [46]:
def build_change_impact_prompt(
    changes,
    customer_need,
    appointment,
    previous_commitment,
    risk_before,
    risk_after
):
    return f"""
ROLE: Recall service change impact reviewer.

CONFIRMED CHANGE
{json.dumps(changes, indent=2)}

CURRENT CONTEXT
- Customer need: {customer_need}
- Appointment: {appointment}
- Previous customer commitment: {previous_commitment}
- Rule-based risk before: {risk_before}
- Rule-based risk after: {risk_after}

TASK
Explain which existing customer expectation or workflow is affected by the confirmed change.

CONSTRAINTS
1. Do not recalculate or change the risk values.
2. Do not change database values.
3. Do not create a Next Action.
4. Do not invent facts outside the provided context.
5. Support the impact with evidence from the provided facts.

OUTPUT JSON ONLY
{{
  "change_summary": "string",
  "affected_customer_expectation": "string or null",
  "affected_appointment": true/false,
  "workflow_impact": "string or null",
  "evidence_facts": ["string"]
}}
"""

In [47]:
change_impact_prompt = build_change_impact_prompt(
    changes=changes,
    customer_need="2026-09-03",
    appointment="2026-09-03",
    previous_commitment="The part should arrive before Sep 3.",
    risk_before=risk_before,
    risk_after=risk_after
)

print(change_impact_prompt)


ROLE: Recall service change impact reviewer.

CONFIRMED CHANGE
{
  "eta": {
    "before": "2026-09-02",
    "after": "2026-09-06"
  }
}

CURRENT CONTEXT
- Customer need: 2026-09-03
- Appointment: 2026-09-03
- Previous customer commitment: The part should arrive before Sep 3.
- Rule-based risk before: WATCH
- Rule-based risk after: AT_RISK

TASK
Explain which existing customer expectation or workflow is affected by the confirmed change.

CONSTRAINTS
1. Do not recalculate or change the risk values.
2. Do not change database values.
3. Do not create a Next Action.
4. Do not invent facts outside the provided context.
5. Support the impact with evidence from the provided facts.

OUTPUT JSON ONLY
{
  "change_summary": "string",
  "affected_customer_expectation": "string or null",
  "affected_appointment": true/false,
  "workflow_impact": "string or null",
  "evidence_facts": ["string"]
}



In [48]:
raw_change_impact = call_model(change_impact_prompt)
print(raw_change_impact)

{"change_summary":"The confirmed ETA moved from 2026-09-02 to 2026-09-06.","affected_customer_expectation":"The previous commitment that the part should arrive before Sep 3 is no longer supported by the confirmed ETA.","affected_appointment":true,"workflow_impact":"The part is now expected after the 2026-09-03 appointment, affecting the appointment workflow that depends on its arrival.","evidence_facts":["Customer need: 2026-09-03.","Appointment: 2026-09-03.","Previous customer commitment: The part should arrive before Sep 3.","ETA changed from 2026-09-02 to 2026-09-06.","Rule-based risk changed from WATCH to AT_RISK."]}


In [49]:
parsed_change_impact, change_impact_error = extract_json(raw_change_impact)

print("Parsed:")
print(parsed_change_impact)

print("\nParse Error:")
print(change_impact_error)

Parsed:
{'change_summary': 'The confirmed ETA moved from 2026-09-02 to 2026-09-06.', 'affected_customer_expectation': 'The previous commitment that the part should arrive before Sep 3 is no longer supported by the confirmed ETA.', 'affected_appointment': True, 'workflow_impact': 'The part is now expected after the 2026-09-03 appointment, affecting the appointment workflow that depends on its arrival.', 'evidence_facts': ['Customer need: 2026-09-03.', 'Appointment: 2026-09-03.', 'Previous customer commitment: The part should arrive before Sep 3.', 'ETA changed from 2026-09-02 to 2026-09-06.', 'Rule-based risk changed from WATCH to AT_RISK.']}

Parse Error:
None


## Step 4. Inspect Change Impact Evidence

JSON Parsing 성공 여부만으로 기능을 성공으로 판단하지 않는다.

다음 항목을 사람이 직접 확인한다.

1. ETA `Sep 2 → Sep 6` 변경을 정확히 설명했는가?
2. 기존 고객 기대 `"part should arrive before Sep 3"`를 영향 대상으로 찾았는가?
3. Sep 3 Appointment를 영향 대상으로 인식했는가?
4. 입력에 없는 사실을 추가하지 않았는가?
5. Rule이 계산한 `WATCH → AT_RISK`를 임의로 변경하지 않았는가?
6. Next Action을 생성하지 않았는가?
7. 결과의 근거가 `evidence_facts`에 남아 있는가?

In [50]:
print(
    json.dumps(
        parsed_change_impact,
        indent=2,
        ensure_ascii=False
    )
)

{
  "change_summary": "The confirmed ETA moved from 2026-09-02 to 2026-09-06.",
  "affected_customer_expectation": "The previous commitment that the part should arrive before Sep 3 is no longer supported by the confirmed ETA.",
  "affected_appointment": true,
  "workflow_impact": "The part is now expected after the 2026-09-03 appointment, affecting the appointment workflow that depends on its arrival.",
  "evidence_facts": [
    "Customer need: 2026-09-03.",
    "Appointment: 2026-09-03.",
    "Previous customer commitment: The part should arrive before Sep 3.",
    "ETA changed from 2026-09-02 to 2026-09-06.",
    "Rule-based risk changed from WATCH to AT_RISK."
  ]
}


## Step 5. Observation & Evaluation

### Result

ETA가 `2026-09-02 → 2026-09-06`으로 지연된 Case에서
Change Impact Radar가 기존 고객 기대와 Appointment에 대한 영향을 구조화했다.

- Confirmed Change: ETA Sep 2 → Sep 6
- Rule Risk: `WATCH → AT_RISK`
- Previous Customer Commitment: `"The part should arrive before Sep 3."`
- Appointment Impact: `true`
- JSON Parsing: Success

### Human Evaluation

1. ETA 변경을 정확히 설명했다.
2. 기존 고객 약속이 영향을 받는다는 점을 확인했다.
3. Sep 3 Appointment를 영향 대상으로 인식했다.
4. 입력 Facts 밖의 새로운 사실을 생성하지 않았다.
5. Rule이 계산한 `WATCH → AT_RISK`를 임의로 변경하지 않았다.
6. Next Action을 생성하지 않았다.
7. 판단에 사용한 근거를 `evidence_facts`로 반환했다.

### Key Design Decision

Change Impact Radar에서는 LLM에게 위험도를 결정하게 하지 않았다.

Python Rule이 날짜와 상태를 비교해 Risk를 계산하고,
LLM은 이미 확인된 변경이 고객 기대와 Workflow에 어떤 의미를 갖는지만 설명한다.

### Limitation

현재 결과는 ETA 지연이 기존 고객 약속 및 Appointment와 충돌하는
하나의 통제 Scenario에 대한 Prototype 검증이다.

## Step 6. Harmless Change Test

첫 Scenario는 ETA 지연으로 실제 고객 약속과 Appointment에 영향이 생기는 Case였다.

이번에는 반대로 ETA가 더 빨라지는 변경을 테스트한다.

### Scenario

- Before ETA: Sep 2
- After ETA: Sep 1
- Customer Need: Sep 3
- Appointment: Sep 3
- Previous Commitment: `"The part should arrive before Sep 3."`

### Expected Behavior

- Rule Risk는 `WATCH → WATCH`로 유지되어야 한다.
- 기존 고객 약속과 충돌하지 않아야 한다.
- Sep 3 Appointment를 위험 상황으로 과잉 해석하지 않아야 한다.
- LLM은 새로운 Risk나 Next Action을 생성하지 않아야 한다.

**Test Goal:**  
위험한 변경뿐 아니라 안전한 변경도 구분하여 False Positive를 줄일 수 있는지 확인한다.

In [51]:
before_safe = {
    "eta": "2026-09-02",
    "part_status": "IN_TRANSIT"
}

after_safe = {
    "eta": "2026-09-01",
    "part_status": "IN_TRANSIT"
}

changes_safe = {
    key: {
        "before": before_safe.get(key),
        "after": after_safe.get(key)
    }
    for key in after_safe
    if before_safe.get(key) != after_safe.get(key)
}

risk_before_safe = promise_risk(
    part_status="IN_TRANSIT",
    eta="2026-09-02",
    promised_date="2026-09-03"
)

risk_after_safe = promise_risk(
    part_status="IN_TRANSIT",
    eta="2026-09-01",
    promised_date="2026-09-03"
)

print("Changes:", changes_safe)
print("Before:", risk_before_safe)
print("After :", risk_after_safe)

Changes: {'eta': {'before': '2026-09-02', 'after': '2026-09-01'}}
Before: WATCH
After : WATCH


### Rule Check

먼저 LLM 호출 전에 Python Rule 결과를 확인한다.

ETA가 더 빨라졌더라도 Part가 아직 `IN_TRANSIT`이므로
물리 입고가 확정된 것은 아니다.

따라서 Risk를 `SAFE`로 낮추지 않고 `WATCH`를 유지하는 것이 현재 Rule의 기준이다.

이 단계에서도 Risk 판단은 LLM이 아니라 Code가 수행한다.

In [52]:
change_impact_prompt_safe = build_change_impact_prompt(
    changes=changes_safe,
    customer_need="2026-09-03",
    appointment="2026-09-03",
    previous_commitment="The part should arrive before Sep 3.",
    risk_before=risk_before_safe,
    risk_after=risk_after_safe
)

raw_change_impact_safe = call_model(change_impact_prompt_safe)

parsed_change_impact_safe, change_impact_safe_error = extract_json(
    raw_change_impact_safe
)

print(
    json.dumps(
        parsed_change_impact_safe,
        indent=2,
        ensure_ascii=False
    )
)

print("\nParse Error:")
print(change_impact_safe_error)

{
  "change_summary": "The confirmed ETA moved one day earlier, from 2026-09-02 to 2026-09-01.",
  "affected_customer_expectation": "The expectation that the part should arrive before Sep 3 remains supported, now with an additional day before the customer need and appointment.",
  "affected_appointment": false,
  "workflow_impact": "No appointment disruption is indicated; the updated ETA remains before the 2026-09-03 appointment.",
  "evidence_facts": [
    "Previous customer commitment: The part should arrive before Sep 3.",
    "ETA changed from 2026-09-02 to 2026-09-01.",
    "Customer need is 2026-09-03.",
    "Appointment is 2026-09-03.",
    "Rule-based risk remains WATCH before and after the change."
  ]
}

Parse Error:
None


## Step 7. Harmless Change Observation

### Result

ETA가 `2026-09-02 → 2026-09-01`로 앞당겨진 Scenario에서는
기존 고객 약속과 Appointment에 부정적인 영향이 없다고 판단했다.

- Rule Risk: `WATCH → WATCH`
- Previous Commitment: `"The part should arrive before Sep 3."`
- Appointment Impact: `false`
- Workflow Disruption: 없음
- JSON Parsing: Success

### Human Evaluation

1. ETA가 하루 빨라진 변경을 정확히 설명했다.
2. 기존 `"before Sep 3"` 고객 기대가 여전히 충족된다고 판단했다.
3. Sep 3 Appointment를 영향 대상으로 과잉 탐지하지 않았다.
4. Rule이 계산한 `WATCH → WATCH`를 그대로 유지했다.
5. 새로운 Risk나 Next Action을 생성하지 않았다.
6. 판단 근거를 `evidence_facts`로 반환했다.

### Key Finding

Change Impact Radar는 위험한 변경만 탐지하는 것이 아니라,
고객 약속에 영향을 주지 않는 변경에서는 불필요한 경고를 생성하지 않았다.

### Limitation

현재 검증은 ETA 지연 1건과 ETA 개선 1건의 통제 Scenario를 대상으로 한다.
실제 운영 데이터 전체에 대한 성능을 의미하지 않는다.

## Step 8. Change Impact Evaluation Summary

두 가지 반대 Scenario를 비교한다.

| Scenario | ETA Change | Rule Result | Expected Impact |
|---|---|---|---|
| CI1 Delay | Sep 2 → Sep 6 | WATCH → AT_RISK | Customer expectation and Appointment affected |
| CI2 Improvement | Sep 2 → Sep 1 | WATCH → WATCH | No Appointment disruption |

이 실험의 핵심은 LLM이 Risk를 계산하는 것이 아니라,
Code가 계산한 Risk와 Confirmed Change를 바탕으로
고객 기대와 Workflow에 미치는 의미만 설명하도록 역할을 제한한 것이다.

In [53]:
change_impact_eval = pd.DataFrame([
    {
        "case_id": "CI1",
        "scenario": "ETA delay",
        "eta_before": "2026-09-02",
        "eta_after": "2026-09-06",
        "risk_before": risk_before,
        "risk_after": risk_after,
        "expected_appointment_impact": True,
        "actual_appointment_impact": parsed_change_impact["affected_appointment"],
        "expected_behavior_met": parsed_change_impact["affected_appointment"] is True,
        "json_valid": change_impact_error is None
    },
    {
        "case_id": "CI2",
        "scenario": "ETA improvement",
        "eta_before": "2026-09-02",
        "eta_after": "2026-09-01",
        "risk_before": risk_before_safe,
        "risk_after": risk_after_safe,
        "expected_appointment_impact": False,
        "actual_appointment_impact": parsed_change_impact_safe["affected_appointment"],
        "expected_behavior_met": parsed_change_impact_safe["affected_appointment"] is False,
        "json_valid": change_impact_safe_error is None
    }
])

change_impact_eval

,case_id,scenario,eta_before,eta_after,risk_before,risk_after,expected_appointment_impact,actual_appointment_impact,expected_behavior_met,json_valid
0,CI1,ETA delay,2026-09-02,2026-09-06,WATCH,AT_RISK,True,True,True,True
1,CI2,ETA improvement,2026-09-02,2026-09-01,WATCH,WATCH,False,False,True,True


### Feature 3 — Initial Evaluation Result

두 개의 반대 Scenario에서 사전에 정의한 기대 동작을 모두 확인했다.

- **CI1 — ETA Delay**
  - ETA: Sep 2 → Sep 6
  - Rule: WATCH → AT_RISK
  - Appointment Impact: `true`
  - Expected Behavior Met: `true`

- **CI2 — ETA Improvement**
  - ETA: Sep 2 → Sep 1
  - Rule: WATCH → WATCH
  - Appointment Impact: `false`
  - Expected Behavior Met: `true`

### What This Validated

Change Impact Radar는 ETA 변경 자체를 LLM에게 판단시키지 않았다.

1. Code가 Before / After 차이를 계산했다.
2. Rule이 공급 Risk를 계산했다.
3. LLM은 확인된 변경이 기존 고객 약속과 Appointment에 미치는 의미만 설명했다.
4. 안전한 변경에서는 불필요한 Appointment Risk를 생성하지 않았다.

### Limitation

현재 결과는 ETA Delay 1건과 ETA Improvement 1건의 통제 Scenario에 한정된 초기 Prototype 검증이다.

In [54]:
change_impact_eval.to_csv(
    "change_impact_human_evaluation.csv",
    index=False
)

change_impact_results = {
    "CI1_delay": parsed_change_impact,
    "CI2_improvement": parsed_change_impact_safe
}

with open("change_impact_results.json", "w") as f:
    json.dump(change_impact_results, f, indent=2)

with open("change_impact_prompt.txt", "w") as f:
    f.write(
        build_change_impact_prompt(
            changes={"eta": {
                "before": "{BEFORE_ETA}",
                "after": "{AFTER_ETA}"
            }},
            customer_need="{CUSTOMER_NEED}",
            appointment="{APPOINTMENT}",
            previous_commitment="{PREVIOUS_COMMITMENT}",
            risk_before="{RULE_RISK_BEFORE}",
            risk_after="{RULE_RISK_AFTER}"
        )
    )

print("Saved Change Impact Radar files")

Saved Change Impact Radar files


In [55]:
change_impact_files = [
    "change_impact_human_evaluation.csv",
    "change_impact_results.json",
    "change_impact_prompt.txt"
]

with zipfile.ZipFile("recallrelay_change_impact.zip", "w") as zipf:
    for file in change_impact_files:
        zipf.write(file)

files.download("recallrelay_change_impact.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

# Feature 4 — Next Best Question

## Problem

Recall Case에서 필요한 정보가 부족한 상태인데도
AI가 상황을 추측해서 답을 만들어내면 잘못된 고객 안내로 이어질 수 있다.

Next Best Question의 목적은 정답을 생성하는 것이 아니라,
**현재 판단에 필요한 정보 중 무엇이 빠져 있는지를 먼저 확인하게 하는 것**이다.

### Design Principle

**Code / Rule**
- 필수 Field의 누락 여부 확인
- 어떤 정보가 필요한지 Missing Code 생성

**LLM**
- 왜 해당 정보를 먼저 확인해야 하는지 직원에게 설명
- 확인해야 할 질문을 이해하기 쉬운 문장으로 표현

### Boundary

- LLM은 누락된 값을 추측하지 않는다.
- 새로운 ETA나 고객 요구일을 생성하지 않는다.
- DB 값을 수정하지 않는다.
- 정보가 부족하면 부족하다고 명시한다.

## Step 1. Detect Missing Information with Code

AI를 호출하기 전에 일반 코드로 Missing Field를 먼저 확인한다.

### Initial Test Scenarios

- **NBQ1:** Part=`IN_TRANSIT`, ETA=`None`
  - Expected Missing: `LATEST_PART_ETA`

- **NBQ2:** Customer Requested Date=`None`
  - Expected Missing: `CUSTOMER_REQUESTED_DATE`

- **NBQ3:** 필요한 정보가 모두 존재
  - Expected Missing: 없음

이 단계의 핵심은 "무엇이 빠졌는가"를 LLM 판단에 맡기지 않는 것이다.

In [56]:
def find_missing_information(part_status, eta, requested_date):
    missing = []

    if part_status in ["ORDERED", "IN_TRANSIT"] and not eta:
        missing.append("LATEST_PART_ETA")

    if not requested_date:
        missing.append("CUSTOMER_REQUESTED_DATE")

    return missing

## Step 2. Rule-only Missing Field Test

먼저 LLM을 호출하지 않고 세 개의 Scenario에서
Code가 예상한 Missing Field를 정확히 찾는지 확인한다.

### Expected

- NBQ1 → `LATEST_PART_ETA`
- NBQ2 → `CUSTOMER_REQUESTED_DATE`
- NBQ3 → Missing 없음

이 단계가 맞아야 이후 LLM에게 설명 역할을 맡길 수 있다.

In [57]:
nbq_cases = [
    {
        "case_id": "NBQ1",
        "part_status": "IN_TRANSIT",
        "eta": None,
        "requested_date": "2026-09-03",
        "expected_missing": ["LATEST_PART_ETA"]
    },
    {
        "case_id": "NBQ2",
        "part_status": "IN_TRANSIT",
        "eta": "2026-09-02",
        "requested_date": None,
        "expected_missing": ["CUSTOMER_REQUESTED_DATE"]
    },
    {
        "case_id": "NBQ3",
        "part_status": "IN_TRANSIT",
        "eta": "2026-09-02",
        "requested_date": "2026-09-03",
        "expected_missing": []
    }
]

nbq_rule_results = []

for case in nbq_cases:
    actual_missing = find_missing_information(
        part_status=case["part_status"],
        eta=case["eta"],
        requested_date=case["requested_date"]
    )

    nbq_rule_results.append({
        "case_id": case["case_id"],
        "expected_missing": case["expected_missing"],
        "actual_missing": actual_missing,
        "expected_behavior_met": actual_missing == case["expected_missing"]
    })

nbq_rule_df = pd.DataFrame(nbq_rule_results)
nbq_rule_df

,case_id,expected_missing,actual_missing,expected_behavior_met
0,NBQ1,[LATEST_PART_ETA],[LATEST_PART_ETA],True
1,NBQ2,[CUSTOMER_REQUESTED_DATE],[CUSTOMER_REQUESTED_DATE],True
2,NBQ3,[],[],True


## Step 3. Explain What to Verify First with LLM

Missing Field 자체는 이미 Code가 확정했다.

이제 LLM에는 누락된 값을 추측하게 하지 않고,
직원이 왜 해당 정보를 먼저 확인해야 하는지 설명하는 역할만 맡긴다.

### LLM Boundary

- 누락된 ETA를 생성하지 않는다.
- 고객 Requested Date를 추측하지 않는다.
- DB 값을 변경하지 않는다.
- Next Action을 임의로 결정하지 않는다.
- Missing Field가 없으면 불필요한 질문을 만들지 않는다.

In [63]:
def build_next_best_question_prompt(
    part_status,
    eta,
    requested_date,
    missing_fields
):
    return f"""
ROLE: Recall service information-gap reviewer.

CURRENT FACTS
- Part status: {part_status}
- ETA: {eta}
- Customer requested date: {requested_date}

CODE-DETECTED MISSING FIELDS
{json.dumps(missing_fields)}

TASK
Explain what information should be verified first and why it is needed.

CONSTRAINTS
1. Do not invent missing values.
2. Do not create an ETA or customer requested date.
3. Do not change database values.
4. Do not create or change operational status.
5. If no required information is missing, do not invent a question.

OUTPUT JSON ONLY
{{
  "information_missing": true/false,
  "verify_first": "string or null",
  "reason": "string or null",
  "missing_fields": ["string"],
  "evidence_facts": ["string"]
}}
"""

## Step 4. Run LLM Explanation Test

Code가 이미 Missing Field를 확정한 상태에서
LLM에게는 누락된 값을 채우게 하지 않고,
직원이 무엇을 먼저 확인해야 하는지 설명하게 한다.

### Test Focus

- NBQ1: ETA를 새로 만들어내지 않고 `LATEST_PART_ETA` 확인을 요구하는가?
- NBQ2: Requested Date를 추측하지 않고 `CUSTOMER_REQUESTED_DATE` 확인을 요구하는가?
- NBQ3: Missing Field가 없을 때 불필요한 질문을 만들지 않는가?

이 단계에서는 "AI가 정답을 만들어냈는가"보다
**정보가 부족할 때 추측하지 않았는가**를 더 중요하게 본다.

In [60]:
nbq_llm_results = []

for case in nbq_cases:
    missing_fields = find_missing_information(
        part_status=case["part_status"],
        eta=case["eta"],
        requested_date=case["requested_date"]
    )

    prompt = build_next_best_question_prompt(
        part_status=case["part_status"],
        eta=case["eta"],
        requested_date=case["requested_date"],
        missing_fields=missing_fields
    )

    raw = call_model(prompt)
    parsed, err = extract_json(raw)

    nbq_llm_results.append({
        "case_id": case["case_id"],
        "missing_fields": missing_fields,
        "raw_response": raw,
        "parsed": parsed,
        "json_valid": err is None,
        "parse_error": err
    })

## Step 5. Inspect Raw LLM Results

JSON Parsing 성공 여부만으로 기능을 성공으로 판단하지 않는다.

다음 세 가지를 특히 확인한다.

**NBQ1**
`ETA=None`인데 모델이 임의의 ETA를 만들면 실패다.

**NBQ2**
`requested_date=None`인데 모델이 고객 요청 날짜를 추측하면 실패다.

**NBQ3**
필요한 정보가 이미 모두 있는데도 새로운 질문을 만들면 False Positive다.

또한 세 Case 모두 DB 값, Operational Status, Next Action을 임의로 변경하면 안 된다.

In [61]:
for result in nbq_llm_results:
    print("=" * 70)
    print("CASE:", result["case_id"])
    print("CODE-DETECTED MISSING:", result["missing_fields"])

    print(
        json.dumps(
            result["parsed"],
            indent=2,
            ensure_ascii=False
        )
    )

    print("JSON VALID:", result["json_valid"])
    print()

CASE: NBQ1
CODE-DETECTED MISSING: ['LATEST_PART_ETA']
{
  "information_missing": true,
  "verify_first": "Verify the latest part ETA with the shipment or supplier source.",
  "reason": "The part is IN_TRANSIT, but no ETA is recorded. The latest part ETA is needed to assess expected arrival against the customer requested date without estimating or altering any values.",
  "missing_fields": [
    "LATEST_PART_ETA"
  ],
  "evidence_facts": [
    "Part status: IN_TRANSIT",
    "ETA: None",
    "Customer requested date: 2026-09-03"
  ]
}
JSON VALID: True

CASE: NBQ2
CODE-DETECTED MISSING: ['CUSTOMER_REQUESTED_DATE']
{
  "information_missing": true,
  "verify_first": "Verify whether the customer has a requested delivery date and, if so, obtain that date.",
  "reason": "The customer requested date is missing and is needed to compare the customer's requested timing with the current ETA.",
  "missing_fields": [
    "CUSTOMER_REQUESTED_DATE"
  ],
  "evidence_facts": [
    "Part status is IN_TRAN

## Step 6. Initial Observation

### What Worked

- **NBQ1**
  - `LATEST_PART_ETA` 누락을 인식했다.
  - ETA 값을 임의로 생성하지 않았다.
  - `information_missing: true`로 정보 부족을 명시했다.

- **NBQ2**
  - `CUSTOMER_REQUESTED_DATE` 누락을 인식했다.
  - 고객 요청 날짜를 임의로 추측하지 않았다.

- **NBQ3**
  - Missing Field가 없을 때 `information_missing: false`를 반환했다.
  - `verify_first: null`로 불필요한 질문을 생성하지 않았다.

- 세 Case 모두 JSON Parsing에 성공했다.

### Minor Prompt Quality Issues

초기 결과에서 기능 자체는 동작했지만 두 가지 표현 범위 문제가 확인되었다.

1. NBQ1에서 `"shipment or supplier source"`라는 확인 출처를 입력 Facts에 없는데도 제안했다.
2. NBQ2에서 `CUSTOMER_REQUESTED_DATE`를 `"requested delivery date"`라고 바꿔 표현했다.

값을 Hallucination한 것은 아니지만,
RecallRelay의 실제 업무 용어와 입력 범위를 더 엄격하게 유지하기 위해 Prompt를 한 번 더 제한한다.

### Next Improvement

- Missing Field의 의미만 설명한다.
- 입력에 없는 조회 시스템, 부서, 공급처를 제안하지 않는다.
- Field 이름의 업무 의미를 임의로 다른 용어로 바꾸지 않는다.

## Step 7. Prompt Refinement — Terminology & Scope Guardrail

초기 Prompt는 Missing Field를 올바르게 설명했지만,
Raw Response를 검토하면서 두 가지 범위 문제가 발견되었다.

1. `LATEST_PART_ETA` 설명에서 입력에 없던 `"shipment or supplier source"`를 제안했다.
2. `CUSTOMER_REQUESTED_DATE`를 `"requested delivery date"`로 바꿔 표현했다.

값 자체를 만들어낸 것은 아니지만,
RecallRelay의 실제 업무 용어와 입력 범위를 더 엄격하게 유지할 필요가 있다고 판단했다.

### Improvement

- Code가 탐지한 Missing Field만 설명
- Field Definition 명시
- 입력에 없는 시스템 / 공급처 / 부서 / 조회 출처 제안 금지
- Recall Service 업무 용어 유지
- Missing Field가 없으면 질문 생성 금지

목표는 더 많은 답을 생성하는 것이 아니라,
**LLM의 설명 범위를 Code가 확인한 정보 안으로 좁히는 것**이다.

In [64]:
def build_next_best_question_prompt_v2(
    part_status,
    eta,
    requested_date,
    missing_fields
):
    return f"""
ROLE: Recall service information-gap reviewer.

CURRENT FACTS
- Part status: {part_status}
- ETA: {eta}
- Customer requested date: {requested_date}

CODE-DETECTED MISSING FIELDS
{json.dumps(missing_fields)}

FIELD DEFINITIONS
- LATEST_PART_ETA: latest expected arrival date for the recall part
- CUSTOMER_REQUESTED_DATE: date requested by the customer for recall service

TASK
Explain which code-detected missing information should be verified first
and why it is needed for the current recall case.

CONSTRAINTS
1. Do not invent missing values.
2. Do not create an ETA or customer requested date.
3. Use only the CODE-DETECTED MISSING FIELDS provided above.
4. Do not suggest a system, supplier, department, or source unless it is provided in CURRENT FACTS.
5. Preserve the business meaning of the field definitions.
6. Do not change database values or operational status.
7. Do not create a Next Action.
8. If CODE-DETECTED MISSING FIELDS is empty, return no verification question.

OUTPUT JSON ONLY
{{
  "information_missing": true/false,
  "verify_first": "string or null",
  "reason": "string or null",
  "missing_fields": ["string"],
  "evidence_facts": ["string"]
}}
"""

## Step 8. Re-test with Refined Prompt v2

동일한 NBQ1, NBQ2, NBQ3 Scenario를 유지하고
Prompt만 v1에서 v2로 변경해 다시 실행한다.

### 확인할 것

- NBQ1에서 입력에 없는 `supplier`, `shipment source` 등을 더 이상 제안하지 않는가?
- NBQ2에서 `CUSTOMER_REQUESTED_DATE`의 Recall Service 의미를 유지하는가?
- NBQ3에서 Missing Field가 없을 때 계속 `verify_first: null`을 반환하는가?
- 누락된 값 자체를 추측하지 않는가?
- JSON 구조가 유지되는가?

다른 조건은 유지하고 Prompt Guardrail의 효과만 비교한다.

In [65]:
nbq_llm_results_v2 = []

for case in nbq_cases:
    missing_fields = find_missing_information(
        part_status=case["part_status"],
        eta=case["eta"],
        requested_date=case["requested_date"]
    )

    prompt = build_next_best_question_prompt_v2(
        part_status=case["part_status"],
        eta=case["eta"],
        requested_date=case["requested_date"],
        missing_fields=missing_fields
    )

    raw = call_model(prompt)
    parsed, err = extract_json(raw)

    nbq_llm_results_v2.append({
        "case_id": case["case_id"],
        "missing_fields": missing_fields,
        "raw_response": raw,
        "parsed": parsed,
        "json_valid": err is None,
        "parse_error": err
    })

## Step 9. Inspect Refined Prompt Results

Prompt v2가 단순히 다른 문장을 생성했는지가 아니라,
v1에서 발견된 범위 문제가 실제로 줄었는지 확인한다.

### v1 → v2 Comparison

**NBQ1**
- v1: 입력에 없던 조회 출처를 제안
- v2 목표: `LATEST_PART_ETA` 자체만 확인하도록 제한

**NBQ2**
- v1: `CUSTOMER_REQUESTED_DATE`를 delivery date로 바꿔 표현
- v2 목표: Recall Service의 Customer Requested Date 의미 유지

**NBQ3**
- v1: 불필요한 질문 없음
- v2 목표: 동일 동작 유지

In [66]:
for result in nbq_llm_results_v2:
    print("=" * 70)
    print("CASE:", result["case_id"])
    print("CODE-DETECTED MISSING:", result["missing_fields"])

    print(
        json.dumps(
            result["parsed"],
            indent=2,
            ensure_ascii=False
        )
    )

    print("JSON VALID:", result["json_valid"])
    print()

CASE: NBQ1
CODE-DETECTED MISSING: ['LATEST_PART_ETA']
{
  "information_missing": true,
  "verify_first": "What is the latest expected arrival date for the recall part?",
  "reason": "LATEST_PART_ETA should be verified first because the part is IN_TRANSIT but no ETA is recorded. The latest expected arrival date is needed to assess timing against the customer's requested recall service date of 2026-09-03.",
  "missing_fields": [
    "LATEST_PART_ETA"
  ],
  "evidence_facts": [
    "Part status: IN_TRANSIT",
    "ETA: None",
    "Customer requested date: 2026-09-03"
  ]
}
JSON VALID: True

CASE: NBQ2
CODE-DETECTED MISSING: ['CUSTOMER_REQUESTED_DATE']
{
  "information_missing": true,
  "verify_first": "What date, if any, has the customer requested for recall service?",
  "reason": "CUSTOMER_REQUESTED_DATE is needed to determine the customer's preferred recall service date in relation to the current part status and ETA.",
  "missing_fields": [
    "CUSTOMER_REQUESTED_DATE"
  ],
  "evidence_

## Step 10. v1 vs v2 Final Evaluation

### Refined Prompt Result

동일한 NBQ1, NBQ2, NBQ3 Scenario를 Prompt v2로 다시 실행한 결과,
v1에서 확인된 표현 범위 문제가 개선되었다.

### NBQ1 — Missing Part ETA

- Code-detected Missing Field: `LATEST_PART_ETA`
- ETA 값을 생성하지 않았다.
- v1에서 나타났던 입력에 없는 `supplier / shipment source` 제안이 사라졌다.
- Missing Field 자체의 의미만 설명했다.
- JSON Parsing: Success

### NBQ2 — Missing Customer Requested Date

- Code-detected Missing Field: `CUSTOMER_REQUESTED_DATE`
- 고객 요청 날짜를 추측하지 않았다.
- v1의 `"requested delivery date"` 표현 대신 Recall Service 문맥의 Customer Requested Date 의미를 유지했다.
- JSON Parsing: Success

### NBQ3 — No Missing Information

- `information_missing: false`
- `verify_first: null`
- Missing Field가 없는 상태에서 불필요한 질문을 생성하지 않았다.
- JSON Parsing: Success

### What Changed from v1 to v2

Prompt v2에 Field Definition과 Scope Guardrail을 추가해
LLM이 Code가 확인한 Missing Field 밖으로 설명을 확장하지 못하도록 제한했다.

### Key Design Decision

Next Best Question에서 중요한 것은 AI가 더 많은 답을 만드는 것이 아니다.

Code가 정보 부족을 먼저 판정하고,
LLM은 **이미 확인된 Missing Field의 의미만 설명**하도록 역할을 제한했다.

### Limitation

현재 결과는 3개의 통제 Scenario를 대상으로 한 초기 Prototype 검증이며,
실제 운영 환경 전체에 대한 성능을 의미하지 않는다.

In [67]:
nbq_human_eval = pd.DataFrame([
    {
        "case_id": "NBQ1",
        "prompt_version": "v1",
        "missing_field_correct": True,
        "invented_value": False,
        "scope_guardrail_met": False,
        "terminology_preserved": True,
        "false_positive_question": False,
        "json_valid": True,
        "notes": "Correct missing field, but suggested an unsupported shipment/supplier source."
    },
    {
        "case_id": "NBQ2",
        "prompt_version": "v1",
        "missing_field_correct": True,
        "invented_value": False,
        "scope_guardrail_met": True,
        "terminology_preserved": False,
        "false_positive_question": False,
        "json_valid": True,
        "notes": "Correct missing field, but rephrased CUSTOMER_REQUESTED_DATE as a delivery date."
    },
    {
        "case_id": "NBQ3",
        "prompt_version": "v1",
        "missing_field_correct": True,
        "invented_value": False,
        "scope_guardrail_met": True,
        "terminology_preserved": True,
        "false_positive_question": False,
        "json_valid": True,
        "notes": "Correctly returned no verification question."
    },

    {
        "case_id": "NBQ1",
        "prompt_version": "v2",
        "missing_field_correct": True,
        "invented_value": False,
        "scope_guardrail_met": True,
        "terminology_preserved": True,
        "false_positive_question": False,
        "json_valid": True,
        "notes": "Explained LATEST_PART_ETA without introducing an unsupported lookup source."
    },
    {
        "case_id": "NBQ2",
        "prompt_version": "v2",
        "missing_field_correct": True,
        "invented_value": False,
        "scope_guardrail_met": True,
        "terminology_preserved": True,
        "false_positive_question": False,
        "json_valid": True,
        "notes": "Preserved the recall-service meaning of CUSTOMER_REQUESTED_DATE."
    },
    {
        "case_id": "NBQ3",
        "prompt_version": "v2",
        "missing_field_correct": True,
        "invented_value": False,
        "scope_guardrail_met": True,
        "terminology_preserved": True,
        "false_positive_question": False,
        "json_valid": True,
        "notes": "Correctly returned no verification question."
    }
])

nbq_human_eval

,case_id,prompt_version,missing_field_correct,invented_value,scope_guardrail_met,terminology_preserved,false_positive_question,json_valid,notes
0,NBQ1,v1,True,False,False,True,False,True,"Correct missing field, but suggested an unsupported shipment/supplier source."
1,NBQ2,v1,True,False,True,False,False,True,"Correct missing field, but rephrased CUSTOMER_REQUESTED_DATE as a delivery date."
2,NBQ3,v1,True,False,True,True,False,True,Correctly returned no verification question.
3,NBQ1,v2,True,False,True,True,False,True,Explained LATEST_PART_ETA without introducing an unsupported lookup source.
4,NBQ2,v2,True,False,True,True,False,True,Preserved the recall-service meaning of CUSTOMER_REQUESTED_DATE.
5,NBQ3,v2,True,False,True,True,False,True,Correctly returned no verification question.


In [68]:
nbq_summary = nbq_human_eval.groupby("prompt_version").agg(
    missing_field_accuracy=("missing_field_correct", "mean"),
    scope_guardrail_rate=("scope_guardrail_met", "mean"),
    terminology_preservation_rate=("terminology_preserved", "mean"),
    json_valid_rate=("json_valid", "mean")
)

nbq_summary = (nbq_summary * 100).round(1)
nbq_summary

,missing_field_accuracy,scope_guardrail_rate,terminology_preservation_rate,json_valid_rate
prompt_version,,,,
v1,100.0,66.7,66.7,100.0
v2,100.0,100.0,100.0,100.0


### Initial Comparison Summary

Prompt v1과 v2 모두 Missing Field 자체는 정확하게 유지했지만,
v2에서는 추가 Guardrail을 통해 설명 범위를 더 엄격하게 제한했다.

- Missing Field detection: Code에서 수행
- Missing value invention: 없음
- Scope / terminology 문제: v1에서 발견 → v2에서 개선
- No-missing Case의 불필요한 질문: 없음
- JSON Parsing: 유지

이 비교는 Prompt를 더 길게 만든 것이 목적이 아니라,
AI가 업무 범위를 벗어나지 않도록 제약을 구체화한 과정이다.

In [69]:
nbq_rule_df.to_csv(
    "next_best_question_rule_evaluation.csv",
    index=False
)

nbq_human_eval.to_csv(
    "next_best_question_human_evaluation.csv",
    index=False
)

nbq_summary.to_csv(
    "next_best_question_prompt_summary.csv"
)

nbq_results = {
    "v1": {
        result["case_id"]: result["parsed"]
        for result in nbq_llm_results
    },
    "v2": {
        result["case_id"]: result["parsed"]
        for result in nbq_llm_results_v2
    }
}

with open("next_best_question_results.json", "w") as f:
    json.dump(nbq_results, f, indent=2)

print("Saved Next Best Question evaluation files")

Saved Next Best Question evaluation files


In [70]:
with open("next_best_question_prompt_v1.txt", "w") as f:
    f.write(
        build_next_best_question_prompt(
            part_status="{PART_STATUS}",
            eta="{ETA}",
            requested_date="{CUSTOMER_REQUESTED_DATE}",
            missing_fields=["{CODE_DETECTED_MISSING_FIELD}"]
        )
    )

with open("next_best_question_prompt_v2.txt", "w") as f:
    f.write(
        build_next_best_question_prompt_v2(
            part_status="{PART_STATUS}",
            eta="{ETA}",
            requested_date="{CUSTOMER_REQUESTED_DATE}",
            missing_fields=["{CODE_DETECTED_MISSING_FIELD}"]
        )
    )

print("Saved NBQ prompt v1 and v2")

Saved NBQ prompt v1 and v2


## Feature 4 — Evidence Saved

Next Best Question 실험에서 다음 증거를 저장했다.

- Rule-only Missing Field Evaluation
- Prompt v1 / v2 Human Evaluation
- Prompt Version Summary
- Raw LLM Results
- Prompt v1
- Refined Prompt v2

최종 Prompt만 남기지 않고,
초기 결과에서 발견한 Scope / Terminology 문제와 수정 이후 결과를 함께 보존한다.

In [71]:
nbq_files = [
    "next_best_question_rule_evaluation.csv",
    "next_best_question_human_evaluation.csv",
    "next_best_question_prompt_summary.csv",
    "next_best_question_results.json",
    "next_best_question_prompt_v1.txt",
    "next_best_question_prompt_v2.txt"
]

with zipfile.ZipFile("recallrelay_next_best_question.zip", "w") as zipf:
    for file in nbq_files:
        zipf.write(file)

print("Created recallrelay_next_best_question.zip")

files.download("recallrelay_next_best_question.zip")

Created recallrelay_next_best_question.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

# Feature 5 — Cross-source Contradiction Detector

## Problem

Recall Case의 정보는 Part Record, Interaction Timeline 등 여러 Source에 나뉘어 있다.

예를 들어 현재 Part Record는 `ARRIVED`인데
최근 고객 Interaction에는 `"still in transit"`이라고 남아 있다면
현재 상태와 고객 안내 사이에 실제 충돌이 있을 수 있다.

하지만 과거 Interaction에서 `"still in transit"`이라고 기록된 뒤
나중에 Part가 정상적으로 도착했다면 이것은 모순이 아니라 정상적인 상태 변화다.

### Design Principle

**Code / Rule**
- Source Timestamp 정규화
- 기록의 시간 순서 정렬
- Current Record와 Historical Record 구분

**LLM**
- 시간 관계를 고려해 의미 충돌 여부 설명
- 충돌 근거 Source를 Evidence로 반환

### Boundary

- 오래된 정상 이력을 자동으로 오류라고 판단하지 않는다.
- DB 값을 변경하지 않는다.
- 어느 Source가 맞는지 임의로 확정하지 않는다.
- Next Action을 생성하지 않는다.
- 모순이 있으면 "검토가 필요한 충돌 후보"로만 표시한다.

## Step 1. Ground Truth Contradiction Cases

LLM 결과를 보기 전에 사람이 먼저 기대 결과를 정의한다.

### CC1 — Recent Contradiction

- Part Record: `ARRIVED` at Sep 5 10:00
- Interaction: `"The part is still in transit."` at Sep 5 11:00
- Expected: **Contradiction**

현재 Part Record가 ARRIVED로 갱신된 이후에도
더 늦은 Interaction에서 still in transit이라고 안내했기 때문이다.

### CC2 — Historical State Change

- Interaction: `"The part is still in transit."` at Sep 2 15:00
- Part Record: `ARRIVED` at Sep 5 10:00
- Expected: **No Contradiction**

Interaction이 더 오래된 정상 과거 이력이므로
이후 Part가 도착한 것과 동시에 모순되는 정보가 아니다.

### CC3 — Current Sources Aligned

- Part Record: `ARRIVED` at Sep 5 10:00
- Interaction: `"The part has arrived."` at Sep 5 11:00
- Expected: **No Contradiction**

현재 상태와 고객 Interaction의 의미가 일치한다.

In [72]:
contradiction_cases = [
    {
        "case_id": "CC1",
        "part_record": {
            "timestamp": "2026-09-05T10:00:00",
            "part_status": "ARRIVED"
        },
        "interaction": {
            "timestamp": "2026-09-05T11:00:00",
            "text": "The part is still in transit."
        },
        "expected_contradiction": True
    },
    {
        "case_id": "CC2",
        "part_record": {
            "timestamp": "2026-09-05T10:00:00",
            "part_status": "ARRIVED"
        },
        "interaction": {
            "timestamp": "2026-09-02T15:00:00",
            "text": "The part is still in transit."
        },
        "expected_contradiction": False
    },
    {
        "case_id": "CC3",
        "part_record": {
            "timestamp": "2026-09-05T10:00:00",
            "part_status": "ARRIVED"
        },
        "interaction": {
            "timestamp": "2026-09-05T11:00:00",
            "text": "The part has arrived."
        },
        "expected_contradiction": False
    }
]

## Step 2. Normalize Timestamp with Code

Cross-source 정보는 문장 의미를 비교하기 전에
Timestamp 기준을 먼저 통일한다.

이 단계에서는 LLM을 사용하지 않는다.

### Goal

각 Interaction이 현재 Part Record보다

- 이전 기록인지
- 이후 기록인지
- 동일 시점인지

Code로 먼저 계산한다.

시간 관계가 명확해야
오래된 정상 이력을 현재 모순으로 오인하는 False Positive를 줄일 수 있다.

In [73]:
from datetime import datetime

def normalize_contradiction_case(case):
    part_time = datetime.fromisoformat(
        case["part_record"]["timestamp"]
    )

    interaction_time = datetime.fromisoformat(
        case["interaction"]["timestamp"]
    )

    if interaction_time < part_time:
        temporal_relation = "INTERACTION_BEFORE_PART_RECORD"
    elif interaction_time > part_time:
        temporal_relation = "INTERACTION_AFTER_PART_RECORD"
    else:
        temporal_relation = "SAME_TIME"

    return {
        "case_id": case["case_id"],
        "part_status": case["part_record"]["part_status"],
        "part_record_timestamp": case["part_record"]["timestamp"],
        "interaction_text": case["interaction"]["text"],
        "interaction_timestamp": case["interaction"]["timestamp"],
        "temporal_relation": temporal_relation,
        "expected_contradiction": case["expected_contradiction"]
    }

In [74]:
normalized_contradiction_cases = [
    normalize_contradiction_case(case)
    for case in contradiction_cases
]

contradiction_rule_df = pd.DataFrame(
    normalized_contradiction_cases
)

contradiction_rule_df

,case_id,part_status,part_record_timestamp,interaction_text,interaction_timestamp,temporal_relation,expected_contradiction
0,CC1,ARRIVED,2026-09-05T10:00:00,The part is still in transit.,2026-09-05T11:00:00,INTERACTION_AFTER_PART_RECORD,True
1,CC2,ARRIVED,2026-09-05T10:00:00,The part is still in transit.,2026-09-02T15:00:00,INTERACTION_BEFORE_PART_RECORD,False
2,CC3,ARRIVED,2026-09-05T10:00:00,The part has arrived.,2026-09-05T11:00:00,INTERACTION_AFTER_PART_RECORD,False


## Step 3. Prompt v1 — Temporal Context Included

Timestamp와 시간 관계는 Code가 이미 정규화했다.

첫 Prompt에서는 이 정보를 LLM에 제공하되,
아직 "과거 이력은 정상 상태 변화일 수 있다"는 업무 Guardrail을
구체적인 예시로 가르치지 않는다.

### Test Focus

- CC1의 실제 최근 충돌을 탐지하는가?
- CC2의 오래된 과거 기록을 모순으로 잘못 판단하지 않는가?
- CC3의 일치하는 현재 정보를 정상으로 판단하는가?
- 어느 Source가 맞는지 임의로 확정하지 않는가?
- Next Action을 생성하지 않는가?

In [75]:
def build_contradiction_prompt_v1(case):
    return f"""
ROLE: Recall service cross-source consistency reviewer.

PART RECORD
- Status: {case["part_status"]}
- Timestamp: {case["part_record_timestamp"]}

INTERACTION
- Text: {case["interaction_text"]}
- Timestamp: {case["interaction_timestamp"]}

TEMPORAL RELATION
- {case["temporal_relation"]}

TASK
Determine whether the two sources contain a meaningful contradiction
that should be reviewed.

CONSTRAINTS
1. Do not change database values.
2. Do not decide which source is correct.
3. Do not create a Next Action.
4. Do not invent information outside the provided sources.
5. Return only a contradiction candidate for human review.

OUTPUT JSON ONLY
{{
  "contradiction": true/false,
  "reason": "string or null",
  "source_a": "string",
  "source_b": "string",
  "evidence": ["string"]
}}
"""

## Step 4. Run Initial Contradiction Test

동일한 Prompt v1을 CC1, CC2, CC3에 적용한다.

특히 CC2가 중요하다.

현재 `ARRIVED`와 과거 `"still in transit"`이라는 두 문장만 보면
서로 반대처럼 보이지만,
과거 기록이 현재 상태보다 먼저 발생했다면 정상적인 상태 변화일 수 있다.

따라서 CC2를 Contradiction으로 판단하면
**Temporal False Positive**로 기록한다.

In [76]:
contradiction_results_v1 = []

for case in normalized_contradiction_cases:
    prompt = build_contradiction_prompt_v1(case)

    raw = call_model(prompt)
    parsed, err = extract_json(raw)

    contradiction_results_v1.append({
        "case_id": case["case_id"],
        "expected_contradiction": case["expected_contradiction"],
        "temporal_relation": case["temporal_relation"],
        "raw_response": raw,
        "parsed": parsed,
        "json_valid": err is None,
        "parse_error": err
    })

In [77]:
for result in contradiction_results_v1:
    print("=" * 70)
    print("CASE:", result["case_id"])
    print("EXPECTED:", result["expected_contradiction"])
    print("TEMPORAL:", result["temporal_relation"])

    print(
        json.dumps(
            result["parsed"],
            indent=2,
            ensure_ascii=False
        )
    )

    print("JSON VALID:", result["json_valid"])
    print()

CASE: CC1
EXPECTED: True
TEMPORAL: INTERACTION_AFTER_PART_RECORD
{
  "contradiction": true,
  "reason": "The later interaction states that the part is still in transit, while the earlier part record shows it as arrived.",
  "source_a": "Part record: Status ARRIVED at 2026-09-05T10:00:00",
  "source_b": "Interaction: \"The part is still in transit.\" at 2026-09-05T11:00:00",
  "evidence": [
    "Part record status is ARRIVED.",
    "Interaction occurred after the part record.",
    "The later interaction says the part is still in transit."
  ]
}
JSON VALID: True

CASE: CC2
EXPECTED: False
TEMPORAL: INTERACTION_BEFORE_PART_RECORD
{
  "contradiction": false,
  "reason": null,
  "source_a": "PART RECORD",
  "source_b": "INTERACTION",
  "evidence": [
    "The interaction stated the part was still in transit on 2026-09-02T15:00:00.",
    "The part record showed ARRIVED later, on 2026-09-05T10:00:00."
  ]
}
JSON VALID: True

CASE: CC3
EXPECTED: False
TEMPORAL: INTERACTION_AFTER_PART_RECORD
{


## Step 5. Initial Observation & Evaluation

### Result

3개의 통제 Scenario에서 사전에 정의한 Contradiction 기준과 일치하는 결과를 확인했다.

| Case | Temporal Relation | Expected | Result |
|---|---|---:|---:|
| CC1 | Interaction AFTER Part Record | Contradiction | `true` |
| CC2 | Interaction BEFORE Part Record | No Contradiction | `false` |
| CC3 | Interaction AFTER Part Record | No Contradiction | `false` |

### Human Evaluation

**CC1 — Recent Contradiction**

Part Record가 `ARRIVED`로 갱신된 이후의 Interaction에서
`"The part is still in transit."`이라고 기록되어 실제 충돌 후보로 탐지했다.

**CC2 — Historical State Change**

과거 Interaction의 `"still in transit"` 기록 이후
Part가 `ARRIVED` 상태로 변경된 정상적인 시간 흐름을 인식했다.

문장만 비교하면 서로 반대처럼 보이지만,
Timestamp를 고려해 Contradiction으로 과잉 탐지하지 않았다.

**CC3 — Current Sources Aligned**

현재 Part Record의 `ARRIVED`와 이후 Interaction의
`"The part has arrived."`가 의미적으로 일치하므로
Contradiction을 생성하지 않았다.

### What This Validated

- Timestamp를 고려한 Cross-source 비교가 가능했다.
- 실제 최근 충돌은 탐지했다.
- 오래된 정상 이력을 현재 오류로 오인하는 Temporal False Positive가 발생하지 않았다.
- 어느 Source가 맞는지 임의로 결정하지 않았다.
- Next Action이나 DB 변경을 생성하지 않았다.
- 세 Case 모두 JSON Parsing에 성공했다.

### Key Design Decision

Contradiction Detector는 단순 문자열 불일치 탐지기가 아니다.

Code가 먼저 Source의 시간 관계를 정규화하고,
LLM은 그 Temporal Context 안에서 두 Source가 의미적으로 동시에 성립 가능한지를 검토한다.

결과는 자동 수정이 아니라 **Human Review가 필요한 Contradiction Candidate**로만 사용한다.

### Limitation

현재 결과는 3개의 통제 Scenario에 대한 초기 Prototype 검증이다.
실제 운영 환경의 다양한 Source 조합 전체에 대한 성능을 의미하지 않는다.

In [78]:
contradiction_eval_rows = []

for result in contradiction_results_v1:
    parsed = result["parsed"]
    actual = parsed.get("contradiction") if parsed else None
    expected = result["expected_contradiction"]

    contradiction_eval_rows.append({
        "case_id": result["case_id"],
        "temporal_relation": result["temporal_relation"],
        "expected_contradiction": expected,
        "actual_contradiction": actual,
        "expected_behavior_met": actual == expected,
        "json_valid": result["json_valid"]
    })

contradiction_eval = pd.DataFrame(contradiction_eval_rows)
contradiction_eval

,case_id,temporal_relation,expected_contradiction,actual_contradiction,expected_behavior_met,json_valid
0,CC1,INTERACTION_AFTER_PART_RECORD,True,True,True,True
1,CC2,INTERACTION_BEFORE_PART_RECORD,False,False,True,True
2,CC3,INTERACTION_AFTER_PART_RECORD,False,False,True,True


In [79]:
contradiction_summary = {
    "total_cases": len(contradiction_eval),
    "expected_behavior_met": int(
        contradiction_eval["expected_behavior_met"].sum()
    ),
    "json_valid": int(
        contradiction_eval["json_valid"].sum()
    ),
    "temporal_false_positive": int(
        (
            (contradiction_eval["case_id"] == "CC2") &
            (contradiction_eval["actual_contradiction"] == True)
        ).sum()
    )
}

contradiction_summary

{'total_cases': 3,
 'expected_behavior_met': 3,
 'json_valid': 3,
 'temporal_false_positive': 0}

## Step 6. Feature 5 — Initial Evaluation Result

- Controlled Cases: **3**
- Expected Behavior Met: **3 / 3**
- JSON Parse Success: **3 / 3**
- Temporal False Positive: **0 / 1 historical-state Case**

특히 과거 `"still in transit"` 기록과 이후 `ARRIVED` 상태를
현재 Contradiction으로 잘못 판단하지 않았다는 점을 확인했다.

이 수치는 초기 통제 Test Set에 한정된 결과이며
Production Accuracy를 의미하지 않는다.

In [80]:
contradiction_eval.to_csv(
    "contradiction_human_evaluation.csv",
    index=False
)

contradiction_results = {
    result["case_id"]: result["parsed"]
    for result in contradiction_results_v1
}

with open("contradiction_results.json", "w") as f:
    json.dump(contradiction_results, f, indent=2)

with open("contradiction_summary.json", "w") as f:
    json.dump(contradiction_summary, f, indent=2)

with open("contradiction_detector_prompt_v1.txt", "w") as f:
    sample_case = {
        "part_status": "{PART_STATUS}",
        "part_record_timestamp": "{PART_RECORD_TIMESTAMP}",
        "interaction_text": "{INTERACTION_TEXT}",
        "interaction_timestamp": "{INTERACTION_TIMESTAMP}",
        "temporal_relation": "{CODE_DETECTED_TEMPORAL_RELATION}"
    }

    f.write(build_contradiction_prompt_v1(sample_case))

print("Saved Contradiction Detector files")

Saved Contradiction Detector files


In [81]:
with open("contradiction_ground_truth_cases.json", "w") as f:
    json.dump(contradiction_cases, f, indent=2)

print("Saved contradiction_ground_truth_cases.json")

Saved contradiction_ground_truth_cases.json


In [82]:
contradiction_files = [
    "contradiction_human_evaluation.csv",
    "contradiction_results.json",
    "contradiction_summary.json",
    "contradiction_detector_prompt_v1.txt",
    "contradiction_ground_truth_cases.json"
]

with zipfile.ZipFile(
    "recallrelay_contradiction_detector.zip",
    "w"
) as zipf:
    for file in contradiction_files:
        zipf.write(file)

files.download("recallrelay_contradiction_detector.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>